# Глава 10. Pipeline и кросс-валидация

_Ноутбук сгенерирован из учебника «Машинное обучение для начинающих». Запускайте ячейки по порядку сверху вниз в Google Colab._

**Цель главы  🎯**

После этой главы ты сможешь собрать предобработку и модель в единый Pipeline, применить кросс-валидацию для надёжной оценки и объяснить, почему оценка по одному разбиению ненадёжна.

**Правило курса  🎯**

Начиная с этой главы, все модели в книге собираются через Pipeline. Это профессиональный стандарт, который исключает целый класс ошибок — в первую очередь утечку данных. Дальше без Pipeline мы не работаем — это привычка настоящего специалиста.

## Зачем нужен Pipeline

**Аналогия: конвейер на заводе  💡**

Pipeline — это конвейер на заводе. Данные входят с одного конца, проходят через одни и те же станки в строгом порядке (заполнить пропуски → закодировать текст → отмасштабировать → модель) и выходят готовым предсказанием. Собрав конвейер один раз, ты гарантируешь, что и на обучении, и на тесте, и в реальном приложении данные обрабатываются абсолютно одинаково — без ручных ошибок и расхождений.

Помимо порядка и удобства, Pipeline решает главную проблему из главы 6 — утечку данных. Он «настраивает» преобразования (например, вычисляет среднее для заполнения пропусков) ТОЛЬКО на train и затем применяет их к test, а не подсматривает в test заранее. Ручное масштабирование до разделения было источником утечки — Pipeline закрывает эту дыру автоматически.

## Масштабирование признаков

Вспомни проблему KNN из главы 5: возраст 0–80 и цена 0–500 несопоставимы по масштабу, и цена «перевешивает». Масштабирование приводит все признаки к сравнимому диапазону, чтобы ни один не доминировал просто из-за крупных чисел. Самый частый инструмент — StandardScaler, который приводит каждый признак к среднему 0 и типичному разбросу 1.

## Собираем первый Pipeline

In [ ]:
# === Подготовка: выполни первой, если открыл новый блокнот ===
import pandas as pd
from sklearn.model_selection import train_test_split
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
df = pd.read_csv(url).dropna(subset=["bill_length_mm", "body_mass_g", "island", "species"])
X = df[["bill_length_mm", "body_mass_g", "island"]]
y = df["species"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier

# Конвейер из трёх станков, выполняемых по порядку:
pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),   # 1. заполнить пропуски
    ("scale",  StandardScaler()),                    # 2. отмасштабировать
    ("model",  KNeighborsClassifier(n_neighbors=5)), # 3. модель
])

pipe.fit(X_train[["bill_length_mm", "body_mass_g"]], y_train)   # обучаем на числах
print("Pipeline accuracy:", pipe.score(X_test[["bill_length_mm", "body_mass_g"]], y_test))

_Теперь пропуски и масштаб обрабатываются честно, только на train, без утечки._

Красота Pipeline в том, что снаружи он ведёт себя как обычная модель: те же .fit() и .score(). Но внутри выполняет всю цепочку преобразований. Заменить модель или добавить шаг — вопрос одной строки.

## Разные типы столбцов: ColumnTransformer

В реальных данных числа и текст соседствуют: длина клюва (число) и остров (текст «Biscoe»/«Dream»). Числа нужно масштабировать, а текст — превратить в числа (модель не понимает слов). ColumnTransformer позволяет применить разные преобразования к разным столбцам внутри одного конвейера.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

num = ["bill_length_mm", "body_mass_g"]   # числовые столбцы
cat = ["island"]             # текстовые (категориальные) столбцы

prep = ColumnTransformer([
    ("num", StandardScaler(), num),
    ("cat", OneHotEncoder(), cat),   # текст → числа (0/1)
])

full = Pipeline([("prep", prep),
                 ("model", RandomForestClassifier(random_state=42))])

Инструмент OneHotEncoder превращает категорию в набор столбцов из 0 и 1: «Biscoe» → [1, 0, 0], «Dream» → [0, 1, 0]. Так текст становится числами, понятными модели, без выдумывания ложного порядка (ведь один остров не «больше» другого).

## Кросс-валидация: одна проверка — ненадёжно

**Аналогия: одна контрольная или четыре  💡**

Оценить ученика по одной контрольной ненадёжно — вдруг попался неудачный вариант или, наоборот, он списал. Четыре контрольные по разным темам дают честную картину. Кросс-валидация делает то же самое: делит данные на несколько частей (folds), по очереди обучает модель на одних и проверяет на оставшейся, а потом усредняет результаты. Каждый кусок данных успевает побыть и в обучении, и в проверке.

In [ ]:
from sklearn.model_selection import cross_val_score

# cv=5 → делим данные на 5 частей, обучаем и проверяем 5 раз
scores = cross_val_score(full, X, y, cv=5)
print("Оценки по фолдам:", scores.round(3))
print("Среднее:", scores.mean().round(3))

Пять чисел вместо одного показывают не только средний результат, но и его стабильность. Если оценки скачут (0,60 и 0,80) — модель неустойчива, зависит от того, какие данные попали в обучение, и одному запуску верить нельзя. Если все пять близки — результат надёжен. Кросс-валидация — золотой стандарт честной оценки.

## Соревнование класса

Возьмите общий датасет (например, House Prices с Kaggle). Каждый собирает свой Pipeline и настраивает модель, а победитель определяется по средней оценке кросс-валидации — не по одному везучему разбиению. Так соревнование становится по-настоящему честным: побеждает устойчиво хорошая модель, а не удачливая.

## Задания

Базовый уровень

Собери Pipeline с ColumnTransformer на пингвинах (числа bill_length_mm, body_mass_g + категория island). Оцени его через cross_val_score с cv=5. Насколько стабильны пять оценок?

Со звёздочкой  ⭐

Сравни две модели внутри одинакового Pipeline (например, KNN и случайный лес) по средней кросс-валидации. Какая устойчивее (меньше разброс между фолдами)? Почему устойчивость иногда важнее чуть более высокого среднего?

Вопросы для обсуждения  💬

Как именно Pipeline защищает от утечки данных, а ручная обработка — нет?

Почему для KNN масштабирование критично, а для дерева решений — почти не важно?

Чем оценка по кросс-валидации надёжнее одного train/test split?

**Проверь себя  ℹ️**

1. Назови два преимущества Pipeline.

2. Зачем нужен OneHotEncoder?

3. Что показывают пять чисел кросс-валидации, кроме среднего?